# Final Ensemble Training for Sentence Splitting

In this notebook, we train the final **CharCNN + BiLSTM ensemble** for sentence boundary detection in **English** and **Italian**.

After identifying the best ensemble setup, we retrain the base models on the merged data for each language, keep a small validation split for early stopping, and train a **meta-classifier** to combine their predictions for final submission.

In [1]:
!git clone https://github.com/anilegin/nlp-sentenceSplitter.git
!cd nlp-sentenceSplitter/
import sys
sys.path.append('/content/nlp-sentenceSplitter')

Cloning into 'nlp-sentenceSplitter'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 135 (delta 37), reused 109 (delta 11), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 10.20 MiB | 15.78 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [33]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.metrics import classification_report, f1_score

# Import your custom extractors
from utils.featureExtractor import FeatureExtractor
from utils.text_features import TfidfFeatureExtractor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [3]:
data_dir = Path("/content/nlp-sentenceSplitter/data/processed")
datasets = ["UD_English-EWT", "UD_English-GUM", "UD_English-ParTUT", "UD_English-PUD"]

train_dfs, dev_dfs, test_dfs = [], [], []

for ds in datasets:
    prefix = ds.replace("UD_English-", "en_").lower()

    train_path = data_dir / ds / f"{prefix}-ud-train.csv"
    dev_path = data_dir / ds / f"{prefix}-ud-dev.csv"
    test_path = data_dir / ds / f"{prefix}-ud-test.csv"

    if train_path.exists(): train_dfs.append(pd.read_csv(train_path))
    if dev_path.exists(): dev_dfs.append(pd.read_csv(dev_path))
    if test_path.exists(): test_dfs.append(pd.read_csv(test_path))

train_df = pd.concat(train_dfs, ignore_index=True)
dev_df = pd.concat(dev_dfs, ignore_index=True)
test_df = pd.concat(test_dfs, ignore_index=True)

y_train = train_df["label"].astype(int)
y_dev = dev_df["label"].astype(int)
y_test = test_df["label"].astype(int)

print(f"Combined Train shape: {train_df.shape}")
print(f"Combined Dev shape: {dev_df.shape}")
print(f"Combined Test shape: {test_df.shape}")

Combined Train shape: (62663, 9)
Combined Dev shape: (9588, 9)
Combined Test shape: (12574, 9)


## Necessary classes for Dataset and Trainer

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("Extracting numerical features...")
extractor = FeatureExtractor()
X_train_feat = extractor.transform(train_df)
X_dev_feat = extractor.transform(dev_df)
X_test_feat = extractor.transform(test_df)

scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train_feat)
X_dev_num = scaler.transform(X_dev_feat)
X_test_num = scaler.transform(X_test_feat)

print(f"Numerical features shape: {X_train_num.shape}")

Extracting numerical features...
Numerical features shape: (62663, 38)


In [5]:
# defining character Vocabulary
all_text = "".join(train_df["centered_context"].fillna("").values)
chars = sorted(list(set(all_text)))
char2idx = {c: i + 1 for i, c in enumerate(chars)}
char2idx["<UNK>"] = len(char2idx) + 1
vocab_size = len(char2idx) + 1

MAX_LEN = 101 # 50 left + 1 center + 50 right

class SentenceDataset(Dataset):
    def __init__(self, texts, num_features, labels, char2idx, max_len):
        self.texts = texts.fillna("").values
        self.num_features = torch.tensor(num_features, dtype=torch.float32)
        self.labels = torch.tensor(labels.values, dtype=torch.float32)
        self.char2idx = char2idx
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = self.texts[idx]
        seq = [self.char2idx.get(c, self.char2idx["<UNK>"]) for c in text]
        if len(seq) < self.max_len:
            seq = seq + [0] * (self.max_len - len(seq))
        else:
            seq = seq[:self.max_len]

        return torch.tensor(seq, dtype=torch.long), self.num_features[idx], self.labels[idx]

BATCH_SIZE = 128

train_dataset = SentenceDataset(train_df["centered_context"], X_train_num, y_train, char2idx, MAX_LEN)
dev_dataset = SentenceDataset(dev_df["centered_context"], X_dev_num, y_dev, char2idx, MAX_LEN)
test_dataset = SentenceDataset(test_df["centered_context"], X_test_num, y_test, char2idx, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [6]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False

    def __call__(self, val_loss, model, path="best_hybrid_model.pt"):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            torch.save(model.state_dict(), path) # Save best weights
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True

class CharCNNWithFeatures(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_numeric_features,
        embed_dim=64,
        num_filters=128,
        kernel_sizes=(3, 5, 7),
        hidden_dim=128,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k)
            for k in kernel_sizes
        ])

        cnn_dim = num_filters * len(kernel_sizes)

        self.num_proj = nn.Sequential(
            nn.Linear(num_numeric_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.head = nn.Sequential(
            nn.Linear(cnn_dim + hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x_char, x_num):
        emb = self.embedding(x_char)
        emb = emb.transpose(1, 2)

        conv_outs = []
        for conv in self.convs:
            h = torch.relu(conv(emb))
            h = torch.max(h, dim=2).values
            conv_outs.append(h)

        h_char = torch.cat(conv_outs, dim=1)
        h_num = self.num_proj(x_num)

        h = torch.cat([h_char, h_num], dim=1)
        logits = self.head(h).squeeze(1)
        return logits

In [7]:
import statistics
import matplotlib.pyplot as plt

class Trainer():
    """Utility class to train and evaluate a Hybrid Model."""

    def __init__(
        self,
        model,
        loss_function,
        optimizer,
        early_stopper=None
    ):

        self.model = model
        self.loss_function = loss_function
        self.optimizer = optimizer
        self.early_stopper = early_stopper
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def _compute_accuracy_binary(self, predictions, labels):
        # Apply sigmoid because the model outputs raw logits
        probs = torch.sigmoid(predictions)
        discrete_predictions = (probs > 0.5).float()
        return (discrete_predictions.view(-1) == labels.view(-1)).float().mean().item()

    def train(self, train_dataset, valid_dataset, epochs=1):
        assert epochs >= 1 and isinstance(epochs, int)
        print('Training...')

        train_loss = []
        train_accuracy = []
        valid_loss = []
        valid_accuracy = []

        for epoch in range(epochs):
            print(f'\n Epoch {epoch + 1:03d}/{epochs}')

            self.model.train()
            epoch_loss = 0.0
            epoch_accuracy = 0.0

            for step, (x_char, x_num, yb) in enumerate(train_dataset):
                x_char = x_char.to(self.device)
                x_num = x_num.to(self.device)
                yb = yb.to(self.device)

                self.optimizer.zero_grad()
                predictions = self.model(x_char, x_num)

                sample_loss = self.loss_function(predictions, yb)
                sample_loss.backward()
                self.optimizer.step()

                epoch_loss += sample_loss.item()
                epoch_accuracy += self._compute_accuracy_binary(predictions, yb)
                if step % 50 == 0:
                    print(f'    [E: {epoch+1:2d} @ step {step}] current avg loss = {epoch_loss / (step + 1):0.4f}')

            avg_epoch_loss = epoch_loss / len(train_dataset)
            avg_epoch_accuracy = epoch_accuracy / len(train_dataset)

            valid_loss_epoch, valid_accuracy_epoch = self.evaluate(valid_dataset)

            print(f'  [E: {epoch+1:2d}] train loss = {avg_epoch_loss:0.4f} | train accuracy = {avg_epoch_accuracy:0.4f}')
            print(f'  [E: {epoch+1:2d}] valid loss = {valid_loss_epoch:0.4f} | valid accuracy = {valid_accuracy_epoch:0.4f}')

            train_loss.append(avg_epoch_loss)
            train_accuracy.append(avg_epoch_accuracy)
            valid_loss.append(valid_loss_epoch)
            valid_accuracy.append(valid_accuracy_epoch)

            if self.early_stopper:
                self.early_stopper(valid_loss_epoch, self.model, path="best_hybrid_model.pt")
                if self.early_stopper.early_stop:
                    print(f"\nEarly stopping triggered at epoch {epoch+1}! No improvement in validation loss.")
                    break

        print('\n... Done!')

        if self.early_stopper:
            self.model.load_state_dict(torch.load("best_hybrid_model.pt"))
            print("Best model weights restored from Early Stopper.")

        return {
            "avg_epoch_loss": statistics.mean(train_loss),
            "avg_epoch_accuracy": statistics.mean(train_accuracy),
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "valid_loss": valid_loss,
            "valid_accuracy": valid_accuracy
        }

    def evaluate(self, valid_dataset):
        self.model.eval()
        valid_loss = 0.0
        validation_accuracy = 0.0

        with torch.no_grad():
            for x_char, x_num, yb in valid_dataset:
                x_char = x_char.to(self.device)
                x_num = x_num.to(self.device)
                yb = yb.to(self.device)

                predictions = self.model(x_char, x_num)

                sample_loss = self.loss_function(predictions, yb)
                valid_loss += sample_loss.item()
                validation_accuracy += self._compute_accuracy_binary(predictions, yb)

        return valid_loss / len(valid_dataset), validation_accuracy / len(valid_dataset)

In [8]:
def get_predictions(model, data_loader, device):
    model.eval()

    all_probs = []
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x_char, x_num, yb in data_loader:
            x_char = x_char.to(device)
            x_num = x_num.to(device)

            logits = model(x_char, x_num)
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).long()

            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(yb.numpy())

    return np.array(all_probs), np.array(all_preds), np.array(all_labels)

## BiLSTM implementation

In [9]:
class BiLSTMWithFeatures(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_numeric_features,
        embed_dim=64,
        lstm_hidden_dim=128,
        lstm_layers=1,
        hidden_dim=128,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.bilstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=lstm_hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )

        self.num_proj = nn.Sequential(
            nn.Linear(num_numeric_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.head = nn.Sequential(
            nn.Linear((2 * lstm_hidden_dim) + hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x_char, x_num):
        emb = self.embedding(x_char)
        lstm_out, _ = self.bilstm(emb)
        h_char = lstm_out.mean(dim=1)

        h_num = self.num_proj(x_num)
        h = torch.cat([h_char, h_num], dim=1)
        logits = self.head(h).squeeze(1)
        return logits

In [15]:
bilstm_loss_function = nn.BCEWithLogitsLoss()

bilstm_optimizer = torch.optim.Adam(
    bilstm_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

bilstm_early_stopper = EarlyStopping(patience=4, min_delta=1e-4)

bilstm_trainer = Trainer(
    model=bilstm_model,
    loss_function=bilstm_loss_function,
    optimizer=bilstm_optimizer,
    early_stopper=bilstm_early_stopper
)

bilstm_history = bilstm_trainer.train(
    train_dataset=train_loader,
    valid_dataset=dev_loader,
    epochs=50
)

torch.save(bilstm_model.state_dict(), "best_bilstm_with_features.pt")
print("Saved BiLSTM model.")

Training...

 Epoch 001/50
    [E:  1 @ step 0] current avg loss = 0.6898
    [E:  1 @ step 50] current avg loss = 0.3087
    [E:  1 @ step 100] current avg loss = 0.2612
    [E:  1 @ step 150] current avg loss = 0.2396
    [E:  1 @ step 200] current avg loss = 0.2251
    [E:  1 @ step 250] current avg loss = 0.2149
    [E:  1 @ step 300] current avg loss = 0.2070
    [E:  1 @ step 350] current avg loss = 0.1996
    [E:  1 @ step 400] current avg loss = 0.1920
    [E:  1 @ step 450] current avg loss = 0.1855
  [E:  1] train loss = 0.1817 | train accuracy = 0.9350
  [E:  1] valid loss = 0.1294 | valid accuracy = 0.9592

 Epoch 002/50
    [E:  2 @ step 0] current avg loss = 0.1278
    [E:  2 @ step 50] current avg loss = 0.1379
    [E:  2 @ step 100] current avg loss = 0.1373
    [E:  2 @ step 150] current avg loss = 0.1326
    [E:  2 @ step 200] current avg loss = 0.1296
    [E:  2 @ step 250] current avg loss = 0.1274
    [E:  2 @ step 300] current avg loss = 0.1242
    [E:  2 @ step 3

In [16]:
dev_probs_bilstm, dev_preds_bilstm, dev_labels_bilstm = get_predictions(
    bilstm_model,
    dev_loader,
    bilstm_trainer.device
)

print("BiLSTM DEV F1:", f1_score(dev_labels_bilstm, dev_preds_bilstm))
print(classification_report(dev_labels_bilstm, dev_preds_bilstm, digits=4))

test_probs_bilstm, test_preds_bilstm, test_labels_bilstm = get_predictions(
    bilstm_model,
    test_loader,
    bilstm_trainer.device
)

print("BiLSTM TEST F1:", f1_score(test_labels_bilstm, test_preds_bilstm))
print(classification_report(test_labels_bilstm, test_preds_bilstm, digits=4))

BiLSTM DEV F1: 0.9677597134196748
              precision    recall  f1-score   support

         0.0     0.9791    0.9817    0.9804      5951
         1.0     0.9699    0.9656    0.9678      3637

    accuracy                         0.9756      9588
   macro avg     0.9745    0.9737    0.9741      9588
weighted avg     0.9756    0.9756    0.9756      9588

BiLSTM TEST F1: 0.9717662508207485
              precision    recall  f1-score   support

         0.0     0.9838    0.9840    0.9839      8004
         1.0     0.9720    0.9716    0.9718      4570

    accuracy                         0.9795     12574
   macro avg     0.9779    0.9778    0.9778     12574
weighted avg     0.9795    0.9795    0.9795     12574



## CharCNN

In [10]:
class CharCNNWithFeatures(nn.Module):
    def __init__(
        self,
        vocab_size,
        num_numeric_features,
        embed_dim=64,
        num_filters=128,
        kernel_sizes=(3, 5, 7),
        hidden_dim=128,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, kernel_size=k)
            for k in kernel_sizes
        ])

        cnn_dim = num_filters * len(kernel_sizes)

        self.num_proj = nn.Sequential(
            nn.Linear(num_numeric_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.head = nn.Sequential(
            nn.Linear(cnn_dim + hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, x_char, x_num):
        emb = self.embedding(x_char)
        emb = emb.transpose(1, 2)

        conv_outs = []
        for conv in self.convs:
            h = torch.relu(conv(emb))
            h = torch.max(h, dim=2).values
            conv_outs.append(h)

        h_char = torch.cat(conv_outs, dim=1)
        h_num = self.num_proj(x_num)

        h = torch.cat([h_char, h_num], dim=1)
        logits = self.head(h).squeeze(1)
        return logits

In [11]:
charcnn_model = CharCNNWithFeatures(
    vocab_size=vocab_size,
    num_numeric_features=X_train_num.shape[1],
    embed_dim=64,
    num_filters=128,
    kernel_sizes=(3, 5, 7),
    hidden_dim=128,
    dropout=0.3
)

bilstm_model = BiLSTMWithFeatures(
    vocab_size=vocab_size,
    num_numeric_features=X_train_num.shape[1],
    embed_dim=64,
    lstm_hidden_dim=128,
    lstm_layers=1,
    hidden_dim=128,
    dropout=0.3
)

In [13]:
charcnn_loss_function = nn.BCEWithLogitsLoss()

charcnn_optimizer = torch.optim.Adam(
    charcnn_model.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

charcnn_early_stopper = EarlyStopping(patience=4, min_delta=1e-4)

charcnn_trainer = Trainer(
    model=charcnn_model,
    loss_function=charcnn_loss_function,
    optimizer=charcnn_optimizer,
    early_stopper=charcnn_early_stopper
)

charcnn_history = charcnn_trainer.train(
    train_dataset=train_loader,
    valid_dataset=dev_loader,
    epochs=50
)

torch.save(charcnn_model.state_dict(), "best_charcnn_with_features.pt")
print("Saved CharCNN model.")

Training...

 Epoch 001/50
    [E:  1 @ step 0] current avg loss = 0.0285
    [E:  1 @ step 50] current avg loss = 0.0734
    [E:  1 @ step 100] current avg loss = 0.0734
    [E:  1 @ step 150] current avg loss = 0.0708
    [E:  1 @ step 200] current avg loss = 0.0710
    [E:  1 @ step 250] current avg loss = 0.0713
    [E:  1 @ step 300] current avg loss = 0.0727
    [E:  1 @ step 350] current avg loss = 0.0731
    [E:  1 @ step 400] current avg loss = 0.0732
    [E:  1 @ step 450] current avg loss = 0.0738
  [E:  1] train loss = 0.0739 | train accuracy = 0.9711
  [E:  1] valid loss = 0.1056 | valid accuracy = 0.9654

 Epoch 002/50
    [E:  2 @ step 0] current avg loss = 0.0442
    [E:  2 @ step 50] current avg loss = 0.0553
    [E:  2 @ step 100] current avg loss = 0.0551
    [E:  2 @ step 150] current avg loss = 0.0571
    [E:  2 @ step 200] current avg loss = 0.0572
    [E:  2 @ step 250] current avg loss = 0.0588
    [E:  2 @ step 300] current avg loss = 0.0593
    [E:  2 @ step 3

In [17]:
dev_probs_charcnn, dev_preds_charcnn, dev_labels_charcnn = get_predictions(
    charcnn_model,
    dev_loader,
    charcnn_trainer.device
)

print("CharCNN DEV F1:", f1_score(dev_labels_charcnn, dev_preds_charcnn))
print(classification_report(dev_labels_charcnn, dev_preds_charcnn, digits=4))

test_probs_charcnn, test_preds_charcnn, test_labels_charcnn = get_predictions(
    charcnn_model,
    test_loader,
    charcnn_trainer.device
)

print("CharCNN TEST F1:", f1_score(test_labels_charcnn, test_preds_charcnn))
print(classification_report(test_labels_charcnn, test_preds_charcnn, digits=4))

CharCNN DEV F1: 0.9537257094925207
              precision    recall  f1-score   support

         0.0     0.9628    0.9824    0.9725      5951
         1.0     0.9701    0.9379    0.9537      3637

    accuracy                         0.9655      9588
   macro avg     0.9665    0.9601    0.9631      9588
weighted avg     0.9656    0.9655    0.9654      9588

CharCNN TEST F1: 0.9585273896651142
              precision    recall  f1-score   support

         0.0     0.9695    0.9843    0.9768      8004
         1.0     0.9717    0.9457    0.9585      4570

    accuracy                         0.9703     12574
   macro avg     0.9706    0.9650    0.9677     12574
weighted avg     0.9703    0.9703    0.9702     12574



In [18]:
print("CharCNN test size:", len(test_probs_charcnn))
print("BiLSTM test size:", len(test_probs_bilstm))
print("Labels match:", np.array_equal(test_labels_charcnn, test_labels_bilstm))

CharCNN test size: 12574
BiLSTM test size: 12574
Labels match: True


## Meta-Classifier for both of the models

I will use Logistic Regression again for better explainability.

In [19]:
X_meta_dev = np.column_stack([
    dev_probs_charcnn,
    dev_probs_bilstm
])

X_meta_test = np.column_stack([
    test_probs_charcnn,
    test_probs_bilstm
])

y_meta_dev = dev_labels_charcnn
y_meta_test = test_labels_charcnn

print("Meta dev shape:", X_meta_dev.shape)
print("Meta test shape:", X_meta_test.shape)

Meta dev shape: (9588, 2)
Meta test shape: (12574, 2)


In [20]:
from sklearn.linear_model import LogisticRegression

meta_clf = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight="balanced"
)

meta_clf.fit(X_meta_dev, y_meta_dev)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [21]:
meta_test_probs = meta_clf.predict_proba(X_meta_test)[:, 1]
meta_test_preds = (meta_test_probs > 0.5).astype(int)

print("META TEST F1:", f1_score(y_meta_test, meta_test_preds))
print(classification_report(y_meta_test, meta_test_preds, digits=4))

META TEST F1: 0.9725412974510448
              precision    recall  f1-score   support

         0.0     0.9844    0.9843    0.9843      8004
         1.0     0.9724    0.9726    0.9725      4570

    accuracy                         0.9800     12574
   macro avg     0.9784    0.9785    0.9784     12574
weighted avg     0.9800    0.9800    0.9800     12574



Further tuning prediction threshold.

In [22]:
meta_dev_probs = meta_clf.predict_proba(X_meta_dev)[:, 1]

thresholds = np.arange(0.1, 0.91, 0.05)
best_thr = 0.5
best_f1 = -1

for thr in thresholds:
    preds_thr = (meta_dev_probs > thr).astype(int)
    f1 = f1_score(y_meta_dev, preds_thr)
    if f1 > best_f1:
        best_f1 = f1
        best_thr = thr

print("Best meta threshold on dev:", best_thr)
print("Best meta dev F1:", best_f1)

Best meta threshold on dev: 0.7000000000000002
Best meta dev F1: 0.9705963938973647


In [23]:
meta_test_preds_best = (meta_test_probs > best_thr).astype(int)

print("META TEST F1 (tuned threshold):", f1_score(y_meta_test, meta_test_preds_best))
print(classification_report(y_meta_test, meta_test_preds_best, digits=4))

META TEST F1 (tuned threshold): 0.9734259565552983
              precision    recall  f1-score   support

         0.0     0.9807    0.9894    0.9850      8004
         1.0     0.9811    0.9659    0.9734      4570

    accuracy                         0.9808     12574
   macro avg     0.9809    0.9776    0.9792     12574
weighted avg     0.9808    0.9808    0.9808     12574



## Saving meta classifier.

In [ ]:
import joblib

joblib.dump(meta_clf, "meta_logreg_charcnn_bilstm.pkl")
print("Saved meta-classifier.")

## Now given that I have found the best configurations. I will train English and Italian version with all of the data for the submission

In [32]:
data_dir = Path("/content/nlp-sentenceSplitter/data/processed")

english_datasets = [
    "UD_English-EWT",
    "UD_English-GUM",
    "UD_English-ParTUT",
    "UD_English-PUD"
]

en_dfs = []

for ds in english_datasets:
    prefix = ds.replace("UD_English-", "en_").lower()

    train_path = data_dir / ds / f"{prefix}-ud-train.csv"
    dev_path = data_dir / ds / f"{prefix}-ud-dev.csv"
    test_path = data_dir / ds / f"{prefix}-ud-test.csv"

    if train_path.exists():
        en_dfs.append(pd.read_csv(train_path))
    if dev_path.exists():
        en_dfs.append(pd.read_csv(dev_path))
    if test_path.exists():
        en_dfs.append(pd.read_csv(test_path))

english_df = pd.concat(en_dfs, ignore_index=True)
print("English full shape:", english_df.shape)
print(english_df["label"].value_counts())

italian_datasets = [
    "UD_Italian-ISDT",
    "UD_Italian-ParTUT",
    "UD_Italian-PoSTWITA",
    "UD_Italian-VIT"
]

it_dfs = []

for ds in italian_datasets:
    prefix = ds.replace("UD_Italian-", "it_").lower()

    train_path = data_dir / ds / f"{prefix}-ud-train.csv"
    dev_path = data_dir / ds / f"{prefix}-ud-dev.csv"
    test_path = data_dir / ds / f"{prefix}-ud-test.csv"

    if train_path.exists():
        it_dfs.append(pd.read_csv(train_path))
    if dev_path.exists():
        it_dfs.append(pd.read_csv(dev_path))
    if test_path.exists():
        it_dfs.append(pd.read_csv(test_path))

italian_df = pd.concat(it_dfs, ignore_index=True)
print("Italian full shape:", italian_df.shape)
print(italian_df["label"].value_counts())

English full shape: (84825, 9)
label
0    52631
1    32194
Name: count, dtype: int64
Italian full shape: (69162, 9)
label
0    42900
1    26262
Name: count, dtype: int64


In [34]:
english_train_df, english_val_df = train_test_split(
    english_df,
    test_size=0.10,
    random_state=42,
    stratify=english_df["label"]
)

english_train_df = english_train_df.reset_index(drop=True)
english_val_df = english_val_df.reset_index(drop=True)

print("English train:", english_train_df.shape)
print("English val:", english_val_df.shape)
print(english_train_df["label"].value_counts(normalize=True))
print(english_val_df["label"].value_counts(normalize=True))

italian_train_df, italian_val_df = train_test_split(
    italian_df,
    test_size=0.10,
    random_state=42,
    stratify=italian_df["label"]
)

italian_train_df = italian_train_df.reset_index(drop=True)
italian_val_df = italian_val_df.reset_index(drop=True)

print("Italian train:", italian_train_df.shape)
print("Italian val:", italian_val_df.shape)
print(italian_train_df["label"].value_counts(normalize=True))
print(italian_val_df["label"].value_counts(normalize=True))

English train: (76342, 9)
English val: (8483, 9)
label
0    0.620471
1    0.379529
Name: proportion, dtype: float64
label
0    0.620417
1    0.379583
Name: proportion, dtype: float64
Italian train: (62245, 9)
Italian val: (6917, 9)
label
0    0.620291
1    0.379709
Name: proportion, dtype: float64
label
0    0.620211
1    0.379789
Name: proportion, dtype: float64


In [35]:
def prepare_train_val_objects(train_df, val_df, batch_size=128, max_len=101):
    y_train = train_df["label"].astype(int)
    y_val = val_df["label"].astype(int)

    extractor = FeatureExtractor()

    X_train_feat = extractor.transform(train_df)
    X_val_feat = extractor.transform(val_df)

    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train_feat)
    X_val_num = scaler.transform(X_val_feat)

    all_text = "".join(train_df["centered_context"].fillna("").values)
    chars = sorted(list(set(all_text)))
    char2idx = {c: i + 1 for i, c in enumerate(chars)}
    char2idx["<UNK>"] = len(char2idx) + 1
    vocab_size = len(char2idx) + 1

    train_dataset = SentenceDataset(
        texts=train_df["centered_context"],
        num_features=X_train_num,
        labels=y_train,
        char2idx=char2idx,
        max_len=max_len
    )

    val_dataset = SentenceDataset(
        texts=val_df["centered_context"],
        num_features=X_val_num,
        labels=y_val,
        char2idx=char2idx,
        max_len=max_len
    )

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    return {
        "train_df": train_df,
        "val_df": val_df,
        "y_train": y_train,
        "y_val": y_val,
        "X_train_feat": X_train_feat,
        "X_val_feat": X_val_feat,
        "X_train_num": X_train_num,
        "X_val_num": X_val_num,
        "scaler": scaler,
        "char2idx": char2idx,
        "vocab_size": vocab_size,
        "train_dataset": train_dataset,
        "val_dataset": val_dataset,
        "train_loader": train_loader,
        "val_loader": val_loader
    }

In [36]:
english_objects = prepare_train_val_objects(
    english_train_df,
    english_val_df,
    batch_size=128,
    max_len=101
)

print("English vocab size:", english_objects["vocab_size"])
print("English numeric feature dim:", english_objects["X_train_num"].shape[1])

italian_objects = prepare_train_val_objects(
    italian_train_df,
    italian_val_df,
    batch_size=128,
    max_len=101
)

print("Italian vocab size:", italian_objects["vocab_size"])
print("Italian numeric feature dim:", italian_objects["X_train_num"].shape[1])

English vocab size: 243
English numeric feature dim: 38
Italian vocab size: 135
Italian numeric feature dim: 38


## English

In [37]:
english_charcnn = CharCNNWithFeatures(
    vocab_size=english_objects["vocab_size"],
    num_numeric_features=english_objects["X_train_num"].shape[1],
    embed_dim=64,
    num_filters=128,
    kernel_sizes=(3, 5, 7),
    hidden_dim=128,
    dropout=0.3
)

english_charcnn_loss = nn.BCEWithLogitsLoss()

english_charcnn_optimizer = torch.optim.Adam(
    english_charcnn.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

english_charcnn_early = EarlyStopping(patience=4, min_delta=1e-4)

english_charcnn_trainer = Trainer(
    model=english_charcnn,
    loss_function=english_charcnn_loss,
    optimizer=english_charcnn_optimizer,
    early_stopper=english_charcnn_early
)

english_charcnn_history = english_charcnn_trainer.train(
    train_dataset=english_objects["train_loader"],
    valid_dataset=english_objects["val_loader"],
    epochs=15
)

Training...

 Epoch 001/15
    [E:  1 @ step 0] current avg loss = 0.6626
    [E:  1 @ step 50] current avg loss = 0.4088
    [E:  1 @ step 100] current avg loss = 0.3182
    [E:  1 @ step 150] current avg loss = 0.2715
    [E:  1 @ step 200] current avg loss = 0.2483
    [E:  1 @ step 250] current avg loss = 0.2305
    [E:  1 @ step 300] current avg loss = 0.2177
    [E:  1 @ step 350] current avg loss = 0.2063
    [E:  1 @ step 400] current avg loss = 0.1991
    [E:  1 @ step 450] current avg loss = 0.1931
    [E:  1 @ step 500] current avg loss = 0.1883
    [E:  1 @ step 550] current avg loss = 0.1837
  [E:  1] train loss = 0.1796 | train accuracy = 0.9358
  [E:  1] valid loss = 0.1212 | valid accuracy = 0.9548

 Epoch 002/15
    [E:  2 @ step 0] current avg loss = 0.2168
    [E:  2 @ step 50] current avg loss = 0.1263
    [E:  2 @ step 100] current avg loss = 0.1245
    [E:  2 @ step 150] current avg loss = 0.1224
    [E:  2 @ step 200] current avg loss = 0.1216
    [E:  2 @ step 2

In [38]:
torch.save(english_charcnn.state_dict(), "english_charcnn_final.pt")
joblib.dump(english_objects["char2idx"], "english_char2idx.pkl")
joblib.dump(english_objects["scaler"], "english_scaler.pkl")

english_meta = {
    "max_len": 101,
    "vocab_size": english_objects["vocab_size"],
    "num_numeric_features": english_objects["X_train_num"].shape[1]
}
joblib.dump(english_meta, "english_meta.pkl")

print("Saved English CharCNN + preprocessing objects.")

Saved English CharCNN + preprocessing objects.


In [40]:
english_bilstm = BiLSTMWithFeatures(
    vocab_size=english_objects["vocab_size"],
    num_numeric_features=english_objects["X_train_num"].shape[1],
    embed_dim=64,
    lstm_hidden_dim=128,
    lstm_layers=1,
    hidden_dim=128,
    dropout=0.3
)

english_bilstm_loss = nn.BCEWithLogitsLoss()

english_bilstm_optimizer = torch.optim.Adam(
    english_bilstm.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

english_bilstm_early = EarlyStopping(patience=4, min_delta=1e-4)

english_bilstm_trainer = Trainer(
    model=english_bilstm,
    loss_function=english_bilstm_loss,
    optimizer=english_bilstm_optimizer,
    early_stopper=english_bilstm_early
)

english_bilstm_history = english_bilstm_trainer.train(
    train_dataset=english_objects["train_loader"],
    valid_dataset=english_objects["val_loader"],
    epochs=50
)

Training...

 Epoch 001/50
    [E:  1 @ step 0] current avg loss = 0.6948
    [E:  1 @ step 50] current avg loss = 0.3363
    [E:  1 @ step 100] current avg loss = 0.2748
    [E:  1 @ step 150] current avg loss = 0.2490
    [E:  1 @ step 200] current avg loss = 0.2301
    [E:  1 @ step 250] current avg loss = 0.2162
    [E:  1 @ step 300] current avg loss = 0.2068
    [E:  1 @ step 350] current avg loss = 0.1985
    [E:  1 @ step 400] current avg loss = 0.1912
    [E:  1 @ step 450] current avg loss = 0.1842
    [E:  1 @ step 500] current avg loss = 0.1782
    [E:  1 @ step 550] current avg loss = 0.1730
  [E:  1] train loss = 0.1694 | train accuracy = 0.9389
  [E:  1] valid loss = 0.1186 | valid accuracy = 0.9601

 Epoch 002/50
    [E:  2 @ step 0] current avg loss = 0.1505
    [E:  2 @ step 50] current avg loss = 0.1452
    [E:  2 @ step 100] current avg loss = 0.1382
    [E:  2 @ step 150] current avg loss = 0.1368
    [E:  2 @ step 200] current avg loss = 0.1327
    [E:  2 @ step 2

In [41]:
torch.save(english_bilstm.state_dict(), "english_bilstm_final.pt")
print("Saved English BiLSTM.")

Saved English BiLSTM.


## Italian

In [42]:
italian_charcnn = CharCNNWithFeatures(
    vocab_size=italian_objects["vocab_size"],
    num_numeric_features=italian_objects["X_train_num"].shape[1],
    embed_dim=64,
    num_filters=128,
    kernel_sizes=(3, 5, 7),
    hidden_dim=128,
    dropout=0.3
)

italian_charcnn_loss = nn.BCEWithLogitsLoss()

italian_charcnn_optimizer = torch.optim.Adam(
    italian_charcnn.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

italian_charcnn_early = EarlyStopping(patience=4, min_delta=1e-4)

italian_charcnn_trainer = Trainer(
    model=italian_charcnn,
    loss_function=italian_charcnn_loss,
    optimizer=italian_charcnn_optimizer,
    early_stopper=italian_charcnn_early
)

italian_charcnn_history = italian_charcnn_trainer.train(
    train_dataset=italian_objects["train_loader"],
    valid_dataset=italian_objects["val_loader"],
    epochs=15
)

Training...

 Epoch 001/15
    [E:  1 @ step 0] current avg loss = 0.6976
    [E:  1 @ step 50] current avg loss = 0.3458
    [E:  1 @ step 100] current avg loss = 0.2464
    [E:  1 @ step 150] current avg loss = 0.2104
    [E:  1 @ step 200] current avg loss = 0.1893
    [E:  1 @ step 250] current avg loss = 0.1730
    [E:  1 @ step 300] current avg loss = 0.1629
    [E:  1 @ step 350] current avg loss = 0.1545
    [E:  1 @ step 400] current avg loss = 0.1469
    [E:  1 @ step 450] current avg loss = 0.1421
  [E:  1] train loss = 0.1390 | train accuracy = 0.9546
  [E:  1] valid loss = 0.0996 | valid accuracy = 0.9656

 Epoch 002/15
    [E:  2 @ step 0] current avg loss = 0.1088
    [E:  2 @ step 50] current avg loss = 0.0909
    [E:  2 @ step 100] current avg loss = 0.0901
    [E:  2 @ step 150] current avg loss = 0.0883
    [E:  2 @ step 200] current avg loss = 0.0919
    [E:  2 @ step 250] current avg loss = 0.0919
    [E:  2 @ step 300] current avg loss = 0.0917
    [E:  2 @ step 3

In [43]:
torch.save(italian_charcnn.state_dict(), "italian_charcnn_final.pt")
joblib.dump(italian_objects["char2idx"], "italian_char2idx.pkl")
joblib.dump(italian_objects["scaler"], "italian_scaler.pkl")

italian_meta = {
    "max_len": 101,
    "vocab_size": italian_objects["vocab_size"],
    "num_numeric_features": italian_objects["X_train_num"].shape[1]
}
joblib.dump(italian_meta, "italian_meta.pkl")

print("Saved Italian CharCNN + preprocessing objects.")

Saved Italian CharCNN + preprocessing objects.


In [45]:
italian_bilstm = BiLSTMWithFeatures(
    vocab_size=italian_objects["vocab_size"],
    num_numeric_features=italian_objects["X_train_num"].shape[1],
    embed_dim=64,
    lstm_hidden_dim=128,
    lstm_layers=1,
    hidden_dim=128,
    dropout=0.3
)

italian_bilstm_loss = nn.BCEWithLogitsLoss()

italian_bilstm_optimizer = torch.optim.Adam(
    italian_bilstm.parameters(),
    lr=1e-3,
    weight_decay=1e-5
)

italian_bilstm_early = EarlyStopping(patience=4, min_delta=1e-4)

italian_bilstm_trainer = Trainer(
    model=italian_bilstm,
    loss_function=italian_bilstm_loss,
    optimizer=italian_bilstm_optimizer,
    early_stopper=italian_bilstm_early
)

italian_bilstm_history = italian_bilstm_trainer.train(
    train_dataset=italian_objects["train_loader"],
    valid_dataset=italian_objects["val_loader"],
    epochs=50
)

Training...

 Epoch 001/50
    [E:  1 @ step 0] current avg loss = 0.7200
    [E:  1 @ step 50] current avg loss = 0.2866
    [E:  1 @ step 100] current avg loss = 0.2160
    [E:  1 @ step 150] current avg loss = 0.1857
    [E:  1 @ step 200] current avg loss = 0.1734
    [E:  1 @ step 250] current avg loss = 0.1657
    [E:  1 @ step 300] current avg loss = 0.1599
    [E:  1 @ step 350] current avg loss = 0.1564
    [E:  1 @ step 400] current avg loss = 0.1547
    [E:  1 @ step 450] current avg loss = 0.1509
  [E:  1] train loss = 0.1491 | train accuracy = 0.9609
  [E:  1] valid loss = 0.1293 | valid accuracy = 0.9652

 Epoch 002/50
    [E:  2 @ step 0] current avg loss = 0.1991
    [E:  2 @ step 50] current avg loss = 0.1089
    [E:  2 @ step 100] current avg loss = 0.0953
    [E:  2 @ step 150] current avg loss = 0.0943
    [E:  2 @ step 200] current avg loss = 0.0904
    [E:  2 @ step 250] current avg loss = 0.0916
    [E:  2 @ step 300] current avg loss = 0.0933
    [E:  2 @ step 3

In [46]:
torch.save(italian_bilstm.state_dict(), "italian_bilstm_final.pt")
print("Saved Italian BiLSTM.")

Saved Italian BiLSTM.


In [47]:
en_charcnn_val_probs, en_charcnn_val_preds, en_charcnn_val_labels = get_predictions(
    english_charcnn,
    english_objects["val_loader"],
    english_charcnn_trainer.device
)

print("English CharCNN VAL F1:", f1_score(en_charcnn_val_labels, en_charcnn_val_preds))
print(classification_report(en_charcnn_val_labels, en_charcnn_val_preds, digits=4))

en_bilstm_val_probs, en_bilstm_val_preds, en_bilstm_val_labels = get_predictions(
    english_bilstm,
    english_objects["val_loader"],
    english_bilstm_trainer.device
)

print("English BiLSTM VAL F1:", f1_score(en_bilstm_val_labels, en_bilstm_val_preds))
print(classification_report(en_bilstm_val_labels, en_bilstm_val_preds, digits=4))

English CharCNN VAL F1: 0.9532680770445217
              precision    recall  f1-score   support

         0.0     0.9626    0.9819    0.9722      5263
         1.0     0.9695    0.9376    0.9533      3220

    accuracy                         0.9651      8483
   macro avg     0.9660    0.9598    0.9627      8483
weighted avg     0.9652    0.9651    0.9650      8483

English BiLSTM VAL F1: 0.9727443609022557
              precision    recall  f1-score   support

         0.0     0.9784    0.9888    0.9836      5263
         1.0     0.9814    0.9643    0.9727      3220

    accuracy                         0.9795      8483
   macro avg     0.9799    0.9765    0.9782      8483
weighted avg     0.9795    0.9795    0.9795      8483



In [48]:
it_charcnn_val_probs, it_charcnn_val_preds, it_charcnn_val_labels = get_predictions(
    italian_charcnn,
    italian_objects["val_loader"],
    italian_charcnn_trainer.device
)

print("Italian CharCNN VAL F1:", f1_score(it_charcnn_val_labels, it_charcnn_val_preds))
print(classification_report(it_charcnn_val_labels, it_charcnn_val_preds, digits=4))

it_bilstm_val_probs, it_bilstm_val_preds, it_bilstm_val_labels = get_predictions(
    italian_bilstm,
    italian_objects["val_loader"],
    italian_bilstm_trainer.device
)

print("Italian BiLSTM VAL F1:", f1_score(it_bilstm_val_labels, it_bilstm_val_preds))
print(classification_report(it_bilstm_val_labels, it_bilstm_val_preds, digits=4))

Italian CharCNN VAL F1: 0.9578740157480315
              precision    recall  f1-score   support

         0.0     0.9565    0.9953    0.9756      4290
         1.0     0.9918    0.9262    0.9579      2627

    accuracy                         0.9691      6917
   macro avg     0.9742    0.9607    0.9667      6917
weighted avg     0.9699    0.9691    0.9688      6917

Italian BiLSTM VAL F1: 0.9728166570271833
              precision    recall  f1-score   support

         0.0     0.9761    0.9914    0.9837      4290
         1.0     0.9855    0.9604    0.9728      2627

    accuracy                         0.9796      6917
   macro avg     0.9808    0.9759    0.9783      6917
weighted avg     0.9797    0.9796    0.9796      6917



Final Meta-Classifiers.

In [52]:
X_meta_en = np.column_stack([
    en_charcnn_val_probs,
    en_bilstm_val_probs,
    np.abs(en_charcnn_val_probs - en_bilstm_val_probs),
    (en_charcnn_val_probs + en_bilstm_val_probs) / 2,
    np.maximum(en_charcnn_val_probs, en_bilstm_val_probs),
    np.minimum(en_charcnn_val_probs, en_bilstm_val_probs),
])

y_meta_en = en_charcnn_val_labels

print("English meta feature shape:", X_meta_en.shape)

english_meta_clf = LogisticRegression(
    random_state=42,
    C=0.1,
    max_iter=1000,
    class_weight="balanced"
)

english_meta_clf.fit(X_meta_en, y_meta_en)
print("English meta-classifier trained.")

en_meta_val_probs = english_meta_clf.predict_proba(X_meta_en)[:, 1]
en_meta_val_preds = (en_meta_val_probs > 0.5).astype(int)

print("English META VAL F1:", f1_score(y_meta_en, en_meta_val_preds))
print(classification_report(y_meta_en, en_meta_val_preds, digits=4))

thresholds = np.arange(0.1, 0.91, 0.05)

best_thr_en = 0.5
best_f1_en = -1

for thr in thresholds:
    preds_thr = (en_meta_val_probs > thr).astype(int)
    f1 = f1_score(y_meta_en, preds_thr)
    if f1 > best_f1_en:
        best_f1_en = f1
        best_thr_en = thr

print("Best English meta threshold:", best_thr_en)
print("Best English meta F1:", best_f1_en)

joblib.dump(english_meta_clf, "english_meta_clf.pkl")
joblib.dump(best_thr_en, "english_meta_threshold.pkl")

print("Saved English meta-classifier and threshold.")

English meta feature shape: (8483, 6)
English meta-classifier trained.
English META VAL F1: 0.9731417863835103
              precision    recall  f1-score   support

         0.0     0.9804    0.9871    0.9837      5263
         1.0     0.9786    0.9677    0.9731      3220

    accuracy                         0.9797      8483
   macro avg     0.9795    0.9774    0.9784      8483
weighted avg     0.9797    0.9797    0.9797      8483

Best English meta threshold: 0.7000000000000002
Best English meta F1: 0.9737710067535731
Saved English meta-classifier and threshold.


In [50]:
X_meta_it = np.column_stack([
    it_charcnn_val_probs,
    it_bilstm_val_probs,
    np.abs(it_charcnn_val_probs - it_bilstm_val_probs),
    (it_charcnn_val_probs + it_bilstm_val_probs) / 2,
    np.maximum(it_charcnn_val_probs, it_bilstm_val_probs),
    np.minimum(it_charcnn_val_probs, it_bilstm_val_probs),
])

y_meta_it = it_charcnn_val_labels

print("Italian meta feature shape:", X_meta_it.shape)

italian_meta_clf = LogisticRegression(
    random_state=42,
    C=0.1,
    max_iter=1000,
    class_weight="balanced"
)

italian_meta_clf.fit(X_meta_it, y_meta_it)
print("Italian meta-classifier trained.")

it_meta_val_probs = italian_meta_clf.predict_proba(X_meta_it)[:, 1]
it_meta_val_preds = (it_meta_val_probs > 0.5).astype(int)

print("Italian META VAL F1:", f1_score(y_meta_it, it_meta_val_preds))
print(classification_report(y_meta_it, it_meta_val_preds, digits=4))

best_thr_it = 0.5
best_f1_it = -1

for thr in thresholds:
    preds_thr = (it_meta_val_probs > thr).astype(int)
    f1 = f1_score(y_meta_it, preds_thr)
    if f1 > best_f1_it:
        best_f1_it = f1
        best_thr_it = thr

print("Best Italian meta threshold:", best_thr_it)
print("Best Italian meta F1:", best_f1_it)

joblib.dump(italian_meta_clf, "italian_meta_clf.pkl")
joblib.dump(best_thr_it, "italian_meta_threshold.pkl")

print("Saved Italian meta-classifier and threshold.")

Italian meta feature shape: (6917, 6)
Italian meta-classifier trained.
Italian META VAL F1: 0.9712202609363009
              precision    recall  f1-score   support

         0.0     0.9778    0.9874    0.9826      4290
         1.0     0.9791    0.9635    0.9712      2627

    accuracy                         0.9783      6917
   macro avg     0.9785    0.9754    0.9769      6917
weighted avg     0.9783    0.9783    0.9783      6917

Best Italian meta threshold: 0.6000000000000002
Best Italian meta F1: 0.9718364197530864
Saved Italian meta-classifier and threshold.
